<a href="https://colab.research.google.com/github/SuryamaheshPenke/Gen_AI-Notes/blob/main/Class_4_Tokenization_and_Token_Management.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Class 4 - Tokenization and Token Management**
### Module 1: Prompt Engineering & LLM Mastery
---

## **Agenda**

- Why tokenization exists - models never see words, only integers
- Byte-Pair Encoding (BPE) - how the algorithm builds a vocabulary from scratch
- WordPiece - BERT's approach and how it differs from BPE
- Why the same word costs different tokens in different contexts
- Context window limits - hard limits, soft limits, and the LostInTheMiddle failure
- Token counting and cost estimation with `tiktoken` (no API key needed)
- Prompt compression - removing noise without losing intent

## **PART 1 - Why Tokenization Exists**

A transformer model processes numbers. Not words, not characters, not sentences - integers. Every piece of text that enters a model must first be converted into a sequence of integers, and every piece of output the model generates is produced as a sequence of integers that gets decoded back into text.

This conversion step is tokenization. It sits between your raw text and the model, and it runs on every single API call you make.

The question is not whether to tokenize - that is mandatory. The question is *how* to split text into units, and that design decision has cascading effects on vocabulary size, model size, multilingual performance, and your API bill.

Early NLP systems split text on whitespace (word-level tokenization). This created two problems: the vocabulary grew unbounded as you added more text (every new word needs a new entry), and rare words simply had no representation. Character-level tokenization solved the vocabulary problem but made sequences very long - every character is its own token, and long sequences are expensive to process with attention mechanisms.

Subword tokenization, which is what all modern LLMs use, sits between these extremes. Common words get a single token. Rare words get split into recognizable pieces. The vocabulary stays bounded, and sequences stay manageable.

---

**Simple**

Imagine you are learning a new language and you can only memorize 50,000 vocabulary cards. If you put one word per card, you run out quickly and can't handle new words. If you put one letter per card, you need thousands of cards just to write one sentence. The smart move: put common words on one card, and split rare words into common pieces. "unhappiness" becomes three cards: "un" + "happi" + "ness". You can still read any word, and your 50,000 cards go much further.

That is exactly what a tokenizer does.

**Technical**

A tokenizer maps a string to a sequence of token IDs: `encode(text) -> List[int]`. Each integer is an index into a fixed vocabulary (typically 50k-200k entries for modern models). The vocabulary is built during a separate training phase using a corpus, not during model training itself. The model then learns embeddings for each token ID - a dense vector in high-dimensional space. During inference, the model operates entirely on these integer sequences and their embeddings; it has no access to the original string.

## **PART 2 - Byte-Pair Encoding (BPE)**

BPE started as a data compression algorithm in 1994. Sennrich et al. adapted it for NLP in 2016, and it became the tokenization method for GPT-2, GPT-3, GPT-4, and most OpenAI models.

The core insight: start with individual characters, then repeatedly merge the most frequent adjacent pair into a new symbol. Do this enough times and you have a vocabulary of character n-grams, common words, and subwords - all learned from the training data.

---

**Simple**

You start with the alphabet. You look at your text corpus and ask: what two adjacent letters appear together most often? Say it is "t" and "h". You merge them into a new unit "th" and replace every occurrence of "t h" with "th". Now you have a slightly smaller alphabet plus "th". Repeat: what pair appears most often now? Maybe "th" + "e" = "the". Keep merging until you hit your target vocabulary size.

The result is that common words and common word-pieces get their own tokens. Rare words get split into their most common components.

**Technical**

The BPE training algorithm:
1. Initialize vocabulary with all individual bytes (256 entries for byte-level BPE) or characters.
2. Represent the corpus as sequences of these base units.
3. Count all adjacent pairs across the corpus.
4. Merge the most frequent pair into a new symbol; add it to the vocabulary.
5. Replace all occurrences of that pair in the corpus with the new symbol.
6. Repeat steps 3-5 until vocabulary size reaches the target (e.g., 50,257 for GPT-2).

During encoding (inference time), BPE greedily applies learned merge rules in the order they were learned. The tokenizer maintains a ranked list of merge rules, not just a vocabulary. GPT-4 uses cl100k_base, a byte-level BPE vocabulary with 100,277 tokens.

## **PART 3 - WordPiece (BERT-Style Tokenization)**

WordPiece is the tokenization algorithm used by BERT, RoBERTa, and other encoder-only models from Google. Instead of BPE's frequency-based merging criterion, WordPiece selects merges that maximize the likelihood of the training corpus - a mutual information-style criterion that is slightly more principled but more expensive to train. In practice, WordPiece and BPE produce similar vocabularies. The visible difference you will encounter is the `##` prefix convention: subwords that are not the start of a word are prefixed with `##` (so "tokenization" becomes `token` + `##ization`). BERT uses a 30,000-token WordPiece vocabulary with special tokens `[CLS]` and `[SEP]`.

This course uses BPE/tiktoken models (GPT-4o, Claude, Gemini). You will not work with WordPiece models directly in Module 1, but you will encounter BERT-family models in Module 3 (Class 22 onwards) when building embedding pipelines for RAG.

## **PART 4 - Why the Same Word Costs Different Tokens**

This is where tokenization gets counterintuitive, and it has real cost and performance implications.

---

**Simple**

The tokenizer was trained on a specific corpus. Words and patterns that appeared frequently in that corpus got their own single tokens. Words that appeared rarely got split. This means:

- "cat" is probably one token (very common English word).
- A technical term you invented last month is probably three or four tokens (never seen during tokenizer training).
- A word in Hindi might be six tokens because Hindi was a small fraction of the training corpus.
- The number "1234" might be split differently from "12" or "34" depending on how numbers appeared in training data.

**Technical**

Token cost is determined by vocabulary coverage, which is a function of corpus composition during tokenizer training. For GPT-4's cl100k_base tokenizer:

- English text: roughly 1 token per 4 characters, or 0.75 tokens per word.
- Code: varies significantly - Python keywords are single tokens, variable names split unpredictably.
- Non-Latin scripts (Arabic, Hindi, Chinese): often 2-4x more tokens per word than English equivalents, because the training corpus was English-dominant.
- Numbers: multi-digit integers are often split at arbitrary boundaries (e.g., "2023" may be "20" + "23" or "2" + "023" depending on the tokenizer).
- Whitespace: leading spaces matter. `" cat"` and `"cat"` are different tokens in most GPT tokenizers.

This has a direct pricing implication: a prompt in Tamil costs roughly 3x as many tokens as the same information in English, at the same price per token.

## **PART 5 - Context Window Limits and the LostInTheMiddle Problem**

Every model has a maximum context length - the total number of tokens it can process in one call, counting both input and output. GPT-3.5 had 4,096. GPT-4 Turbo has 128,000. Claude 3 Opus has 200,000.

This limit is real and hard: send more tokens than the context window and the API returns an error. But the soft problem is more dangerous than the hard limit.

---

**Simple**

Imagine you hand someone a 100-page report and ask them a question. They read the first few pages carefully, skim the middle, and read the last few pages carefully. If the answer is on page 47, they might miss it - not because they can't read, but because the middle pages get less attention.

LLMs have a similar pattern. Research shows that information buried in the middle of a long context is retrieved less reliably than information at the beginning or end. This is the LostInTheMiddle problem.

**Technical**

The LostInTheMiddle finding (Liu et al., 2023) showed that multi-document QA performance degrades significantly when the relevant document is placed in the middle of the context rather than at the beginning or end - even for models with large context windows. Performance follows a U-shaped curve: strong at position 0, degrades toward the middle, recovers at the final positions.

Mechanistically, this is attributed to attention sink behavior and positional bias in rotary/ALiBi position encodings. The model attends more strongly to tokens near the start (system prompt, first user message) and near the current generation position.

Practical implications:
- Put the most important instructions at the start and end of your prompt.
- Do not assume a 128k context window means 128k tokens of equal-quality retrieval.
- For retrieval-augmented generation, order retrieved chunks with the most relevant at top and bottom, not randomly.

## **PART 6 - Prompt Compression**

Given that tokens cost money and context windows have quality limits, reducing unnecessary tokens in prompts is a practical engineering skill.

Prompt compression is not about removing information. It is about removing tokens that carry no additional meaning - filler phrases, redundant restatements, verbose formatting instructions that could be shorter.

---

**Simple**

A legal brief has to be precise. A text message to a friend can be sloppy. Most prompts are written more like text messages - with hedging, repetition, and soft phrases that exist to sound polite rather than to convey information. "Please could you kindly help me to summarize the following passage in a clear and concise manner" and "Summarize:" ask for the same thing. One costs 18 tokens, the other costs 2.

Prompt compression is finding all the "please could you kindly" moments and removing them.

**Technical**

Compression techniques fall into three categories:

1. **Lexical compression**: Remove stop words, hedge phrases, verbose format instructions. Achievable manually or with a preprocessing step. Typically 20-30% reduction with no quality loss.

2. **Semantic compression**: Summarize context documents before injecting them into the prompt. If you are doing RAG and injecting 3,000 tokens of retrieved text, a 500-token summary may perform equivalently or better. Requires a fast/cheap model to do the compression.

3. **Learned compression (LLMLingua / similar)**: Train a small model to predict which tokens are essential for a large model to answer a query. Drop low-importance tokens at the byte level. Achieves 40-60% compression with controlled quality degradation. The compressed prompt is not human-readable.

For most production use: lexical compression manually + semantic compression via summarization before RAG injection is sufficient and adds no latency overhead beyond the summarization call.

---
# **Code Demos**

### **Demo 1 - Install tiktoken**

In [ ]:
# tiktoken is OpenAI's tokenizer library - works without any API key
!pip install tiktoken -q

import tiktoken
print("tiktoken version:", tiktoken.__version__)

tiktoken version: 0.13.0


### **Demo 2 - Basic Tokenization: Seeing the Integer Sequence**

In [ ]:
import tiktoken

# cl100k_base is the tokenizer used by GPT-4 and GPT-3.5-turbo
enc = tiktoken.get_encoding("cl100k_base")

text = "Tokenization splits text into pieces a model can process."
tokens = enc.encode(text)

print("Text:", text)
print("Token IDs:", tokens)
print("Token count:", len(tokens))
print()

# Decode each token individually to see the pieces
for token_id in tokens:
    piece = enc.decode([token_id])
    print(f"  {token_id:6d}  ->  {repr(piece)}")

Text: Tokenization splits text into pieces a model can process.
Token IDs: [3404, 2065, 41567, 1495, 1139, 9863, 264, 1646, 649, 1920, 13]
Token count: 11

    3404  ->  'Token'
    2065  ->  'ization'
   41567  ->  ' splits'
    1495  ->  ' text'
    1139  ->  ' into'
    9863  ->  ' pieces'
     264  ->  ' a'
    1646  ->  ' model'
     649  ->  ' can'
    1920  ->  ' process'
      13  ->  '.'


Look at the integer sequence - that is exactly what the model receives. Notice how most common English words are a single token. The space before a word is often *included in that word's token* - `" splits"` is one token, not `" "` + `"splits"`.

### **Demo 3 - Same Information, Different Token Counts (Non-English Cost)**

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# The same sentence in four languages
sentences = {
    "English":  "Artificial intelligence is transforming the world.",
    "Spanish":  "La inteligencia artificial está transformando el mundo.",
    "Hindi":    "कृत्रिम बुद्धिमत्ता दुनिया को बदल रही है।",
    "Arabic":   "الذكاء الاصطناعي يحول العالم.",
}

print(f"{'Language':<10}  {'Tokens':>7}  {'Characters':>12}  {'Tokens/Char':>12}")
print("-" * 48)
for lang, text in sentences.items():
    tokens = enc.encode(text)
    print(f"{lang:<10}  {len(tokens):>7}  {len(text):>12}  {len(tokens)/len(text):>12.2f}")

Language     Tokens    Characters   Tokens/Char
------------------------------------------------
English           8            50          0.16
Spanish          10            55          0.18
Hindi            43            41          1.05
Arabic           20            29          0.69


Hindi and Arabic cost significantly more tokens than English for the same information. The tokenizer was trained on a corpus that was heavily English. Non-Latin scripts get fragmented into many small pieces. If you are building a multilingual product, factor this into your cost estimates - you may pay 3-4x more per message in some languages.

### **Demo 4 - How Numbers Tokenize (Unpredictable Splits)**

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

numbers = ["1", "10", "100", "1000", "10000", "100000",
           "2023", "2024", "3.14159", "0.001", "1,000,000"]

print(f"{'Number':<15}  {'Tokens':>7}  {'Split'}")
print("-" * 50)
for n in numbers:
    token_ids = enc.encode(n)
    pieces = [repr(enc.decode([t])) for t in token_ids]
    print(f"{n:<15}  {len(token_ids):>7}  {' + '.join(pieces)}")

Number tokenization is inconsistent - "1000" and "10000" split at different boundaries. This is why LLMs perform poorly at arithmetic: the model does not see a number, it sees a sequence of subword pieces that happen to look like digits. The model has never learned positional arithmetic on integers.

### **Demo 5 - Context Window: Token Counting Before You Send**

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Model context windows (input + output combined)
MODEL_LIMITS = {
    "gpt-3.5-turbo":  16_385,
    "gpt-4":           8_192,
    "gpt-4-turbo":   128_000,
    "gpt-4o":        128_000,
    "gpt-4o-mini":   128_000,
}

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def check_fits(prompt: str, model: str, max_output_tokens: int = 1000) -> None:
    prompt_tokens = count_tokens(prompt)
    limit = MODEL_LIMITS.get(model, 0)
    remaining = limit - prompt_tokens - max_output_tokens
    status = "OK" if remaining > 0 else "EXCEEDS LIMIT"
    print(f"Model: {model}")
    print(f"  Prompt tokens:      {prompt_tokens:>8,}")
    print(f"  Reserved for output:{max_output_tokens:>8,}")
    print(f"  Context limit:      {limit:>8,}")
    print(f"  Remaining budget:   {remaining:>8,}  [{status}]")
    print()

short_prompt = "Explain transformers in one paragraph."
long_prompt = "Summarize this document: " + ("This is filler text. " * 5000)

check_fits(short_prompt, "gpt-3.5-turbo")
check_fits(long_prompt, "gpt-3.5-turbo")
check_fits(long_prompt, "gpt-4-turbo")

The long prompt exceeds GPT-3.5's limit but fits in GPT-4-turbo. Counting tokens before sending is cheap (runs locally, no API call) and prevents expensive runtime errors. Build this check into any code that constructs dynamic prompts.

### **Demo 6 - Cost Estimation with tiktoken**

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Pricing per 1M tokens (as of mid-2025 - check current pricing before use)
PRICING = {
    "gpt-4o":          {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini":     {"input": 0.15,  "output": 0.60},
    "gpt-4-turbo":     {"input": 10.00, "output": 30.00},
}

def estimate_cost(prompt: str, expected_output_words: int, model: str) -> None:
    input_tokens = len(enc.encode(prompt))
    # Rough estimate: 1 word ~ 1.3 tokens for output
    output_tokens = int(expected_output_words * 1.3)

    prices = PRICING[model]
    input_cost  = (input_tokens  / 1_000_000) * prices["input"]
    output_cost = (output_tokens / 1_000_000) * prices["output"]
    total = input_cost + output_cost

    print(f"Model: {model}")
    print(f"  Input tokens:   {input_tokens:>8,}   cost: ${input_cost:.6f}")
    print(f"  Output tokens:  {output_tokens:>8,}   cost: ${output_cost:.6f}")
    print(f"  Total per call: ${total:.6f}")
    print(f"  Cost per 1000 calls: ${total * 1000:.4f}")
    print()

prompt = """
You are a senior software engineer. Review the following code and identify
bugs, suggest improvements, and explain your reasoning.

" + ("x = x + 1\n" * 200) + "

Provide a detailed code review.
"""

for model in PRICING:
    estimate_cost(prompt, expected_output_words=300, model=model)

The cost difference between gpt-4o and gpt-4o-mini is roughly 15-20x on the same prompt. For a product making 10,000 calls per day, this is the difference between $15/day and $250/day. Always benchmark whether the cheaper model is good enough for your use case before committing to the expensive one.

### **Demo 7 - LostInTheMiddle: Position Matters**

In [ ]:
# This demo simulates the LostInTheMiddle problem without an API call.
# We construct prompts that bury the answer in different positions,
# then show the token cost and position of the relevant content.

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

filler = "This document contains general background information about software engineering practices, team structures, and project management methodologies. It covers topics from agile development to deployment pipelines. " * 10

fact = "The quarterly revenue target for Q3 2025 is $4.2 million."

question = "What is the Q3 2025 revenue target?"

def build_prompt(position: str) -> str:
    if position == "start":
        context = fact + "\n\n" + filler
    elif position == "middle":
        half = len(filler) // 2
        context = filler[:half] + fact + filler[half:]
    else:  # end
        context = filler + "\n\n" + fact
    return f"{context}\n\nQuestion: {question}"

for pos in ["start", "middle", "end"]:
    prompt = build_prompt(pos)
    tokens = enc.encode(prompt)
    total = len(tokens)

    # Find where the fact appears (in tokens)
    fact_tokens = enc.encode(fact)
    fact_str_pos = prompt.find(fact)
    char_fraction = fact_str_pos / len(prompt)

    print(f"Position: {pos.upper()}")
    print(f"  Total tokens:          {total:,}")
    print(f"  Fact location:         {char_fraction:.0%} into the prompt")
    print(f"  Estimated retrieval:   {'HIGH' if pos in ['start', 'end'] else 'DEGRADED (LostInTheMiddle risk)'}")
    print()

When the relevant fact is at 50% into the prompt, retrieval reliability drops - this is the LostInTheMiddle problem from Liu et al. 2023. For RAG pipelines: put the most relevant retrieved chunks at position 0 or at the very end of the context block. Never sort them randomly.

### **Demo 8 - Prompt Compression: Measuring Token Reduction**

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def token_count(text: str) -> int:
    return len(enc.encode(text))

def compression_stats(original: str, compressed: str, label: str) -> None:
    orig_tokens = token_count(original)
    comp_tokens = token_count(compressed)
    reduction = (1 - comp_tokens / orig_tokens) * 100
    print(f"{label}")
    print(f"  Original:   {orig_tokens:>5} tokens")
    print(f"  Compressed: {comp_tokens:>5} tokens")
    print(f"  Reduction:  {reduction:>5.1f}%")
    print()

# Example 1: Verbose vs direct instruction
original_1 = """I would really appreciate it if you could please help me by taking a look at the following text and then providing me with a clear, concise, and well-structured summary that captures all of the main points and key ideas. Please make sure your summary is easy to understand."""

compressed_1 = "Summarize the following text. Capture the main points."

compression_stats(original_1, compressed_1, "Example 1: Verbose instruction")

# Example 2: Over-specified format vs minimal format
original_2 = """Please respond in the following format: First, provide a numbered list of all key points you identify. Second, after the numbered list, write a paragraph that summarizes the overall theme. Third, end with a one-sentence conclusion. Make sure each numbered item is on its own line."""

compressed_2 = "Format: numbered key points, then a summary paragraph, then one-sentence conclusion."

compression_stats(original_2, compressed_2, "Example 2: Over-specified format")

# Example 3: Hedged context vs direct context
original_3 = """I'm not sure if this is the right way to phrase this, but I was wondering if you might be able to assist me with something. I have a Python function that I wrote and I think there might be some issues with it, though I could be wrong. If you have time, could you take a look at it and let me know if you see any bugs or ways it could be improved?"""

compressed_3 = "Review this Python function for bugs and improvements:"

compression_stats(original_3, compressed_3, "Example 3: Hedged request")

The compressed versions are shorter AND clearer. The reduction is 60-75% in each case with no information lost. Hedging, politeness softeners, and redundant format specifications are the main sources of excess tokens in most prompts.

---
# **Key Takeaways**

1. Models receive integer sequences, not text. Tokenization is a mandatory preprocessing step - it converts strings to token ID lists using a fixed vocabulary built from training data.

2. BPE builds vocabulary by repeatedly merging the most frequent adjacent byte pairs. GPT-4 uses cl100k_base, a 100k byte-level BPE vocabulary. WordPiece (BERT) uses a likelihood criterion instead of frequency; the `##` prefix marks continuation subwords.

3. Token cost depends on vocabulary coverage. English text averages 1 token per 4 characters. Non-Latin scripts (Hindi, Arabic, Chinese) can cost 2-4x more tokens than English equivalents because the tokenizer training corpus was English-heavy.

4. Context window limits are both hard (API error above the limit) and soft (LostInTheMiddle: information buried at ~50% of context is retrieved less reliably than information at the start or end).

5. `tiktoken` runs locally with no API key. Use it to count tokens before sending, estimate costs at query time, and validate that your prompt fits the model's limit.

6. Prompt compression of 40-60% is achievable through lexical cleanup (remove hedge phrases, verbose instructions) and semantic compression (summarize injected context before inserting). The compressed prompt is often clearer, not just shorter.

7. Number tokenization is inconsistent and context-dependent - a key reason LLMs are unreliable at arithmetic. The model sees digit subwords, not numeric values.

---
# **Quick Check**

1. You send the same 500-word document to GPT-4 in English and in Hindi. The Hindi version costs about 3x more tokens. Why? What property of the tokenizer causes this?

2. A colleague says "we have a 128k context window so we can safely use 128k tokens of retrieved documents." What is wrong with this reasoning?

3. You are building a RAG pipeline that retrieves 5 documents. Research shows the answer is in document 3 (the middle one). How should you reorder the documents to minimize the LostInTheMiddle risk?

4. What is the difference between BPE and WordPiece in terms of how they select which subword units to create? Which one does GPT-4 use?

5. Why does `tiktoken` not require an API key to count tokens? What is actually happening when you call `enc.encode(text)`?

6. A prompt has this opening sentence: "I would be very grateful if you could please assist me in analyzing the following data and providing your expert insights and recommendations." How many tokens could this be compressed to without losing any actual instruction? Rewrite it.

7. The number "2023" tokenizes differently from "2024" in some tokenizers. Why does this happen, and what does it reveal about how LLMs handle arithmetic?

## **Quick Check - Answers**

---

*Attempt the questions above before reading these answers.*

---

**1.** Tokenizers are trained predominantly on English web text. Hindi and other non-Latin scripts are underrepresented, so fewer merged subword units were learned for them. Each Hindi syllable or character often maps to its own token rather than being merged efficiently - the same meaning requires 3x more tokens than in English.

**2.** Context window is total capacity for input plus output combined. Using 128K tokens of retrieved documents leaves zero tokens for the model's response. The usable retrieval budget is: context window minus system prompt minus space reserved for the answer - typically 50-70% of the nominal window.

**3.** Place the most relevant document first (position 1) and second-most relevant last (position 5). The LostInTheMiddle finding shows models attend most strongly to the beginning and end of a long context. The key document in the middle gets the least attention, so move it to position 1 or 5.

**4.** BPE merges the most frequently co-occurring pair of tokens in the training corpus, iteratively. WordPiece selects merges that maximize the likelihood of the training data - a statistical score, not raw frequency. GPT-4 uses BPE via tiktoken.

**5.** tiktoken uses the model's vocabulary file - a lookup table of subword units and their token IDs - stored locally as a file. Encoding is a deterministic string operation: split text into subwords by matching against the vocabulary, map each subword to an integer ID. No network call is needed.

**6.** Compressed version (about 8 tokens): "Analyze this data and give recommendations." The original is approximately 30 tokens. Every filler phrase ("I would be very grateful", "your expert insights") adds tokens with zero instruction value.

**7.** Tokenizers often assign a single token to number sequences that appear frequently in training data. "2023" may have its own token ID while "2024" is split as "202" + "4" or "20" + "24". This reveals that LLMs process numbers as token sequences, not numerical values - they have no intrinsic numeric understanding, which is why arithmetic errors occur.